# Intro PyTorch: regresión lineal desde cero (réplica)

Objetivo: reconstruir el notebook original pieza por pieza, verificando formas y valores en cada paso.

Plan:
1. Datos sintéticos
2. Forward como función suelta
3. Pérdida y gradiente a mano
4. Un paso de SGD manual y bucle simple
5. Clases `SGD` y `LinearRegression`
6. Minibatches y validación (`TensorDataset`, `DataLoader`)
7. Bucle de entrenamiento completo y gráfica

## 1. Datos sintéticos

**Qué espero:** `X` de forma `(1000, 2)` y `y` de forma `(1000, 1)`, con `y ≈ X @ true_w + true_b`.

Revisar: `.shape`, primeras filas, y la diferencia `y - (X @ w + b)` (debería ser solo ruido).

In [1]:
import torch

# TODO: definir synthetic_regression_data(w, b, num_samples=1000)

# Crear datos sintéticos para probar la infraestructura: 

def synthetic_regression_data(w, b, num_samples = 1000):
    num_inputs = w.shape[0]
    X = torch.randn(num_samples, num_inputs)
    y = X @ w.reshape(-1,1) + b
    y += torch.randn_like(y) * 0.01
    return X, y

    


In [2]:
# TODO: generar X, y con true_w = [2.0, -3.4], true_b = 4.2 e inspeccionar shapes y valores
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2
X, y = synthetic_regression_data(true_w, true_b)

In [3]:
X.shape


torch.Size([1000, 2])

In [4]:
y.shape

torch.Size([1000, 1])

## 2. Forward como función suelta

**Qué espero:** con `w` de forma `(2, 1)` y `b` de forma `(1,)`, `X @ w + b` da `(1000, 1)`. Observar cómo el broadcasting expande `b`.

In [5]:
# TODO: inicializar w (torch.normal) y b (torch.zeros) con requires_grad=True
# TODO: calcular y_hat = X @ w + b y revisar shapes
import torch.nn as nn

w = nn.Parameter(torch.normal(0.0, 0.01, size=(2, 1), requires_grad=True)) ## inicializamos en 0 porque es una regresión lineal y la función de perdida es convexa, entonces siempre ecncontraremos el minimo global)
b = nn.Parameter(torch.zeros(1))

def forward(X, w, b):
    """ Calcula la estimación y_hat dados los paramétros actuales y X"""
    return torch.matmul(X, w) + b

y_hat = forward(X, w, b)



In [6]:
print(w.shape)

print(b.shape)

print(X.shape)

print(y_hat.shape)

torch.Size([2, 1])
torch.Size([1])
torch.Size([1000, 2])
torch.Size([1000, 1])


In [7]:
y_hat

tensor([[-1.8220e-03],
        [ 7.8956e-03],
        [-1.2131e-02],
        [ 4.9423e-03],
        [ 8.1802e-04],
        [-3.0269e-03],
        [-1.7958e-03],
        [ 1.7743e-03],
        [-2.2210e-02],
        [ 8.7471e-03],
        [ 2.3326e-03],
        [ 7.9413e-03],
        [ 1.8865e-02],
        [ 2.6297e-03],
        [ 1.0069e-02],
        [ 1.1543e-02],
        [ 1.7379e-03],
        [ 2.7819e-03],
        [ 3.1695e-03],
        [ 8.6384e-03],
        [ 2.9759e-03],
        [-1.3860e-02],
        [ 4.8311e-03],
        [ 1.7801e-02],
        [-8.0468e-03],
        [-4.4791e-04],
        [ 4.0367e-03],
        [-3.8683e-03],
        [ 1.1502e-02],
        [-1.3537e-02],
        [-3.2073e-03],
        [ 4.1683e-03],
        [ 2.6616e-03],
        [ 6.1963e-03],
        [-5.8768e-04],
        [-7.6396e-03],
        [ 3.8459e-03],
        [ 7.7187e-03],
        [-3.8737e-03],
        [-6.3514e-03],
        [ 5.6893e-03],
        [-5.6222e-03],
        [ 9.3779e-03],
        [-9

## 3. Pérdida y gradiente a mano

**Qué espero:** `loss.backward()` llena `w.grad` (forma `(2, 1)`) y `b.grad` (forma `(1,)`).

Pregunta: ¿por qué el gradiente apunta hacia donde la pérdida *sube*, y por eso restamos?

In [8]:
# TODO: loss = ((y_hat - y) ** 2 / 2).mean()
# TODO: loss.backward() e inspeccionar w.grad, b.grad

def loss(y_hat, y):
    """Calcula la pérdida de regresión lineal
    y devuelve el valor medio sobre el minibatch.
    """
    # 1) Error al cuadrado, escalado por 1/2
    l = (y_hat - y) ** 2 / 2
    # 2) Promedio sobre todas las muestras
    return l.mean()

loss_c = loss(y_hat, y)

loss_c.backward()



In [9]:
loss_c.item()

16.3638916015625

In [10]:
b.grad

tensor([-4.0800])

## 4. Un paso de SGD manual y bucle simple

**Qué espero:** tras `w.data -= lr * w.grad`, la pérdida baja. Repetir en un bucle con todos los datos (sin `DataLoader`) y ver la pérdida decrecer.

Recordar limpiar los gradientes en cada iteración.

In [11]:
# TODO: un solo paso de SGD y recalcular la pérdida

lr = 0.03

w.data -= lr * w.grad

b.data -= lr * b.grad

y_hat = forward(X, w, b)

loss_c4 = loss(y_hat, y)

loss_c4.item()


15.408018112182617

In [19]:
# TODO: bucle de varias iteraciones con todos los datos, imprimiendo la pérdida

params = [w, b]

def zero_grad(params):
    """Pone en cero los gradientes acumulados."""
    for p in params:
        if p.grad is not None:
            p.grad.zero_()

def step(params, lr):
    """Actualiza cada parámetro: θ ← θ − lr * ∇θ."""
    for p in params:
        p.data -= lr * p.grad

num_iters = 100
for i in range(num_iters):
    y_hat = forward(X, w, b)       # 1) predicción con los pesos actuales (w, b sin .data)
    loss_i = loss(y_hat, y)        # 2) pérdida -> crea un grafo nuevo
    zero_grad(params)              # 3) limpia los gradientes de la iteración anterior
    loss_i.backward()              # 4) gradientes nuevos en w.grad y b.grad
    step(params, lr)               # 5) mueve w y b en contra del gradiente
    if i % 30 == 0 or i == num_iters - 1:
        print(f"iter {i:3d} | pérdida {loss_i.item():.6f}")  # 6) monitorear

print("w aprendido:", w.data.reshape(-1))
print("b aprendido:", b.data)

    

iter   0 | pérdida 15.408018
iter  30 | pérdida 2.536195
iter  60 | pérdida 0.418647
iter  90 | pérdida 0.069328
iter  99 | pérdida 0.040457
w aprendido: tensor([ 1.8908, -3.2552])
b aprendido: tensor([3.9892])


## 5. Clases `SGD` y `LinearRegression`

Cada método es algo que ya escribiste suelto arriba:
- `SGD.step` / `SGD.zero_grad` ← pasos 3 y 4
- `LinearRegression.forward` ← paso 2
- `LinearRegression.loss` ← paso 3
- `LinearRegression.configure_optimizers` devuelve un `SGD`

In [13]:
# TODO: class SGD


In [14]:
import torch.nn as nn

# TODO: class LinearRegression(nn.Module)


## 6. Minibatches y validación

**Qué espero:** `train_dataset` con 800 ejemplos y `val_dataset` con 200. Un minibatch de `next(iter(train_loader))` tiene `Xb` de `(32, 2)` y `yb` de `(32, 1)`.

In [15]:
from torch.utils.data import TensorDataset, DataLoader, random_split

# TODO: TensorDataset, random_split 80/20, DataLoaders (batch_size=32, shuffle solo en train)


In [16]:
# TODO: inspeccionar un minibatch con next(iter(train_loader))


## 7. Entrenamiento completo y gráfica

**Qué espero:** ambas pérdidas bajan hacia ~0 (el ruido es 0.01, así que el mínimo teórico ronda 5e-5) y `est_w`, `est_b` quedan muy cerca de los valores verdaderos.

In [17]:
# TODO: instanciar modelo y optimizador
# TODO: bucle por épocas (fase train + fase validación con torch.no_grad())


In [18]:
import matplotlib.pyplot as plt

# TODO: error en w y b, y gráfica de train/val loss por época


## 8. Experimentos (opcional)

Rompe cosas a propósito y anota qué pasa:
- Quitar `zero_grad()`
- `lr = 1.0` y `lr = 0.0001`
- Quitar `shuffle=True`